## FairFlow EDA - German Credit Dataset

### Outputs are saved to ./eda_outputs/

In [51]:
# Import Libraries
import argparse
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
 
warnings.filterwarnings("ignore")

In [52]:
#  Palette & style 
GOOD_COLOR   = "#264653"   # slate blue  → good credit
BAD_COLOR    = "#adb5bd"   # grey    → bad credit
NEUTRAL      = "#F5A623"   # amber accent
BG           = "#FAFAFA"
GRID_COLOR   = "#E8E8E8"
FONT_MAIN    = "DejaVu Sans"
 
PALETTE = {"good": GOOD_COLOR, "bad": BAD_COLOR}
 
plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   BG,
    "axes.edgecolor":   "#CCCCCC",
    "axes.grid":        True,
    "grid.color":       GRID_COLOR,
    "grid.linewidth":   0.6,
    "font.family":      FONT_MAIN,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
})
 
OUT_DIR = "./eda_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
 
def save(fig, name, dpi=180):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f" saved to {path}")
 
 
# Data loader
def load_data(path):
    df = pd.read_csv(path)
    if "Risk" in df.columns:
        df["Risk_binary"] = df["Risk"].str.lower().map({"good": "good", "bad": "bad"})
    elif "risk" in df.columns:
        df["Risk_binary"] = df["risk"].str.lower().map({"good": "good", "bad": "bad"})
 
    # Age groups for fairness analysis
    if "Age" in df.columns and "Age_group" not in df.columns:
        bins   = [17, 25, 35, 45, 100]
        labels = ["18-25", "26-35", "36-45", "46+"]
        df["Age_group"] = pd.cut(df["Age"], bins=bins, labels=labels)
 
    # Debt ratio feature
    if "Credit amount" in df.columns and "Duration" in df.columns:
        df["Debt_ratio"] = df["Credit amount"] / (df["Duration"] + 1)
 
    # Sex_original alias
    if "Sex" in df.columns and "Sex_original" not in df.columns:
        df["Sex_original"] = df["Sex"]
 
    return df

## **EDA 1: Risk by Protected Attributes**


In [53]:
def plot_fairness_overview(df):
    print("[EDA 1] Risk by protected attributes ...")
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Figure 1. Credit Risk Distribution Across Protected Attributes",
                 fontsize=14, fontweight="bold", y=1.01)
 
    for ax, col, title in zip(axes,
                               ["Sex_original", "Age_group"],
                               ["By Sex", "By Age Group"]):
        ct = pd.crosstab(df[col], df["Risk_binary"], normalize="index") * 100
        ct = ct[["good", "bad"]] if "good" in ct.columns else ct
 
        x = np.arange(len(ct))
        w = 0.35
        ax.bar(x - w/2, ct["good"], w, label="Good", color=GOOD_COLOR,
               edgecolor="white", linewidth=1.1)
        ax.bar(x + w/2, ct["bad"],  w, label="Bad",  color=BAD_COLOR,
               edgecolor="white", linewidth=1.1)
 
        # annotate
        for xi, (g, b) in enumerate(zip(ct["good"], ct["bad"])):
            ax.text(xi - w/2, g + 0.8, f"{g:.1f}%", ha="center", fontsize=9, color=GOOD_COLOR, fontweight="bold")
            ax.text(xi + w/2, b + 0.8, f"{b:.1f}%", ha="center", fontsize=9, color=BAD_COLOR,  fontweight="bold")
 
        ax.set_xticks(x)
        ax.set_xticklabels(ct.index, fontsize=11)
        ax.set_ylabel("Percentage (%)", fontsize=11)
        ax.set_title(title, fontsize=12)
        ax.legend(title="Risk", fontsize=10)
        ax.set_ylim(0, 105)
 
        # threshold line at 80% (disparate impact reference)
        ax.axhline(80, color=NEUTRAL, linestyle="--", linewidth=1,
                   label="80% threshold (disparate impact rule)")
 
    plt.tight_layout()
    save(fig, "eda1_risk_by_protected_attributes.png")

## **EDA 2 : Numerical Feature Distribution**


In [54]:
def plot_numerical_distributions(df):
    print("[EDA 2] Numerical feature distributions ...")
    features = ["Age", "Credit amount", "Duration", "Debt_ratio"]
    titles   = ["Age (years)", "Credit Amount (DM)", "Duration (months)", "Debt Ratio (derived)"]
 
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle("Figure 2. Distribution of Drift-Monitored Numerical Features by Risk Class",
                 fontsize=14, fontweight="bold", y=1.01)
 
    for ax, feat, title in zip(axes.flat, features, titles):
        for risk, color in [("good", GOOD_COLOR), ("bad", BAD_COLOR)]:
            subset = df[df["Risk_binary"] == risk][feat].dropna()
            ax.hist(subset, bins=28, alpha=0.35, color=color, density=True, edgecolor="none")
            kde_x = np.linspace(subset.min(), subset.max(), 300)
            kde   = stats.gaussian_kde(subset)
            ax.plot(kde_x, kde(kde_x), color=color, linewidth=2.2,
                    label=f"{risk.capitalize()} (μ={subset.mean():.1f})")
 
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel(feat, fontsize=10)
        ax.set_ylabel("Density", fontsize=10)
        ax.legend(fontsize=9)
 
    plt.tight_layout()
    save(fig, "eda2_numerical_distributions_by_risk.png")

## **EDA 3 : Correlation Heatmap**


In [55]:
def plot_correlation_heatmap(df):
    print("[EDA 3] Correlation heatmap ...")
    num_cols = ["Age", "Job", "Credit amount", "Duration", "Debt_ratio"]
    num_cols = [c for c in num_cols if c in df.columns]
 
    risk_encoded = (df["Risk_binary"] == "bad").astype(int).rename("Risk (bad=1)")
    corr_df      = df[num_cols].join(risk_encoded).corr()
 
    cmap = LinearSegmentedColormap.from_list(
        "fairflow", [GOOD_COLOR, "#FFFFFF", BAD_COLOR], N=256
    )
 
    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
 
    sns.heatmap(corr_df, ax=ax, cmap=cmap, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 11},
                linewidths=0.5, linecolor="white",
                square=True, mask=mask,
                cbar_kws={"shrink": 0.8, "label": "Pearson r"})
 
    ax.set_title("Figure 3. Pearson Correlation Matrix — Numerical Features & Risk",
                 fontsize=13, fontweight="bold", pad=14)
    plt.tight_layout()
    save(fig, "eda3_correlation_heatmap.png")

## **EDA 4 — Categorical features stacked bar breakdown**

In [56]:
def plot_categorical_breakdown(df):
    print("[EDA 4] Categorical feature breakdown ...")
    cats = {
        "Purpose":          "Loan Purpose",
        "Housing":          "Housing Situation",
        "Saving accounts":  "Savings Account Level",
        "Checking account": "Checking Account Level",
    }
    cats = {k: v for k, v in cats.items() if k in df.columns}
 
    n = len(cats)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Figure 4. Categorical Feature Breakdown by Risk Class",
                 fontsize=14, fontweight="bold", y=1.01)
 
    for ax, (col, title) in zip(axes.flat, cats.items()):
        ct = pd.crosstab(df[col], df["Risk_binary"])
        ct = ct.reindex(columns=["good", "bad"])
        ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
        ct_pct = ct_pct.sort_values("bad", ascending=True)
 
        y      = np.arange(len(ct_pct))
        bars_g = ax.barh(y, ct_pct["good"], color=GOOD_COLOR, alpha=0.85,
                          edgecolor="white", linewidth=0.8, label="Good")
        bars_b = ax.barh(y, ct_pct["bad"], left=ct_pct["good"],
                          color=BAD_COLOR, alpha=0.85,
                          edgecolor="white", linewidth=0.8, label="Bad")
 
        ax.set_yticks(y)
        ax.set_yticklabels(ct_pct.index, fontsize=9)
        ax.set_xlim(0, 100)
        ax.set_xlabel("Percentage (%)", fontsize=10)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axvline(70, color="#999", linestyle="--", linewidth=0.8)
        ax.legend(fontsize=9, loc="lower right")
 
    # hide unused subplot if odd number
    for ax in axes.flat[len(cats):]:
        ax.set_visible(False)
 
    plt.tight_layout()
    save(fig, "eda4_categorical_breakdown.png")

## **EDA 5 — Boxplots numerical features by risk (outlier detection)**

In [57]:
def plot_boxplots(df):
    print("[EDA 5] Boxplots ...")
    features = ["Age", "Credit amount", "Duration", "Debt_ratio"]
    titles   = ["Age (years)", "Credit Amount (DM)", "Duration (months)", "Debt Ratio"]
    features = [f for f in features if f in df.columns]
 
    fig, axes = plt.subplots(1, len(features), figsize=(14, 6))
    fig.suptitle("Figure 5. Boxplots of Numerical Features by Risk Class\n(IQR outliers preserved — relevant for drift simulation)",
                 fontsize=13, fontweight="bold", y=1.02)
 
    for ax, feat, title in zip(axes, features, titles):
        data_good = df[df["Risk_binary"] == "good"][feat].dropna()
        data_bad  = df[df["Risk_binary"] == "bad"][feat].dropna()
 
        bp = ax.boxplot([data_good, data_bad],
                        patch_artist=True,
                        widths=0.5,
                        medianprops=dict(color="white", linewidth=2.5),
                        whiskerprops=dict(linewidth=1.2),
                        capprops=dict(linewidth=1.2),
                        flierprops=dict(marker="o", markersize=3, alpha=0.4))
 
        for patch, color in zip(bp["boxes"], [GOOD_COLOR, BAD_COLOR]):
            patch.set_facecolor(color)
            patch.set_alpha(0.8)
 
        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Good", "Bad"], fontsize=11)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_ylabel(feat if feat != "Credit amount" else "Amount (DM)", fontsize=9)
 
    plt.tight_layout()
    save(fig, "eda5_boxplots_by_risk.png")

In [58]:
from pathlib import Path
def main():
    data_path = Path.cwd() / ".." / "data" / "raw" / "german_credit_data.csv"
    df = load_data(data_path)
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}\n")
 
  
    plot_fairness_overview(df)
    plot_numerical_distributions(df)
    plot_correlation_heatmap(df)
    plot_categorical_breakdown(df)
    plot_boxplots(df)
 
    print(f"\nAll EDA figures saved to: {OUT_DIR}/")
 
 
if __name__ == "__main__":
    main()

  Shape: (1000, 15)
  Columns: ['Unnamed: 0', 'Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'Risk', 'Risk_binary', 'Age_group', 'Debt_ratio', 'Sex_original']

[EDA 1] Risk by protected attributes ...
 saved to ./eda_outputs\eda1_risk_by_protected_attributes.png
[EDA 2] Numerical feature distributions ...
 saved to ./eda_outputs\eda2_numerical_distributions_by_risk.png
[EDA 3] Correlation heatmap ...
 saved to ./eda_outputs\eda3_correlation_heatmap.png
[EDA 4] Categorical feature breakdown ...
 saved to ./eda_outputs\eda4_categorical_breakdown.png
[EDA 5] Boxplots ...
 saved to ./eda_outputs\eda5_boxplots_by_risk.png

All EDA figures saved to: ./eda_outputs/
